# Phase 3: Recall Overlay

Question: do manufacturers with FDA recalls on record show up more
often among shortages that turn into permanent discontinuations? The
underlying assumption was that recall trouble is a warning sign. The
data says otherwise - see below.

In [1]:
import sys
sys.path.insert(0, "src")
import pandas as pd
from recall_overlay import (
    build_overlay_dataset, overlap_summary, discontinuation_rate_by_recall,
    scale_check, top_recalled_manufacturers,
)

df = build_overlay_dataset()
summary = overlap_summary(df)
for k, v in summary.items():
    print(f"{k}: {v}")

unique_manufacturers: 145
manufacturers_with_recall_match: 55
rows_total: 1603
rows_with_recall_match: 961


55 of 145 manufacturers on the shortage list (covering 961 of 1,603
rows) match at least one CDER recall record by normalized name.

## Discontinuation rate: manufacturers with vs. without a recall on record

In [2]:
rates, p = discontinuation_rate_by_recall(df, recent_only=False)
print(rates.to_string())
print(f"chi-square p-value: {p:.4f}")

                mean  count
has_recall                 
False       0.320251    637
True        0.249218    959
chi-square p-value: 0.0023


In [3]:
rates_recent, p_recent = discontinuation_rate_by_recall(df, recent_only=True)
print(rates_recent.to_string())
print(f"chi-square p-value: {p_recent:.2e}")

                       mean  count
has_recent_recall                 
False              0.345272    698
True               0.224944    898
chi-square p-value: 1.37e-07


This is the opposite of the original hypothesis. Manufacturers with
any recall on record have a **lower** discontinuation rate (25% vs
32%), and the gap is wider for a recall in the last two years (22% vs
35%) - both statistically significant (p < 0.01 and p < 0.0001).

Before reporting this as a finding, it's worth checking why, since it
contradicts the assumption the phase started with.

## Checking whether this is a manufacturer-scale effect

In [4]:
print("median shortage-list entries per manufacturer, by recall status:")
print(df.groupby("has_recall")["listing_count"].median().to_string())

median shortage-list entries per manufacturer, by recall status:
has_recall
False    16.0
True     42.0


In [5]:
print("same comparison, restricted to manufacturers already at 10+ shortage-list entries:")
print(scale_check(df, min_listings=10).to_string())

same comparison, restricted to manufacturers already at 10+ shortage-list entries:
                  mean  count
recall_count                 
False         0.307159    433
True          0.234266    858


Manufacturers matched to a recall show up on a median of 42
shortage-list entries elsewhere, versus 16 for unmatched manufacturers
- a large difference in scale, using "how many other shortage listings
carry this manufacturer's name" as a stand-in for company size (this
data has no direct revenue or headcount figure).

Restricting to manufacturers already at that larger scale (10+
listings) shrinks the gap from (32% vs 25%) to (31% vs 23%) but
doesn't erase it. So scale explains part of this, not all of it - a
manufacturer under active FDA recall scrutiny may also be under more
pressure to keep supplying a product rather than quietly discontinuing
it, on top of simply being a larger, more resilient supplier to begin
with.

## Who are the most-recalled manufacturers on the shortage list?

In [6]:
print(top_recalled_manufacturers(df, n=10).to_string(index=False))

          mfr_norm  recall_count  recent_recall_count worst_recall_class  listing_count
 accord healthcare           103                   30                  1             26
    sun industries            81                   26                  1             22
          teva usa            55                   22                  1             45
       dr. reddy's            49                   11                  1              1
             lupin            42                   14                  1              7
 baxter healthcare            34                   23                  1             69
            pfizer            32                    6                  1             17
fresenius kabi usa            27                   23                  1            194
   american regent            21                   19                  1              2
            apotex            19                    9                  2             18


Recognizable large generic manufacturers dominate - Accord, Teva,
Baxter, Fresenius Kabi - consistent with the scale explanation above.
Fresenius Kabi in particular shows up on 194 shortage-list entries
(by far the largest footprint in this dataset) with a comparatively
modest 27 recalls.

## Summary

- 55 of 145 shortage-list manufacturers (961 of 1,603 rows) match a
  CDER recall record.
- The original hypothesis - that recall trouble predicts a shortage
  turning into a permanent discontinuation - does not hold. If
  anything, the relationship runs the other way: recall history is
  associated with a lower discontinuation rate, most likely because it
  is entangled with manufacturer scale, and possibly because a
  manufacturer under recall scrutiny has more incentive to keep
  supplying rather than quietly discontinuing.
- Practical takeaway for the Phase 1 risk model: a manufacturer's
  recall history should not be added as a risk-increasing feature -
  the data doesn't support that direction, and using it that way would
  make the tool less accurate, not more.